In [1]:
import pandas as pd  
import numpy as np  
import os  
import re  
import glob  
import io
import calendar
from datetime import datetime  
import msoffcrypto
import warnings
from dateutil.relativedelta import relativedelta
from collections import defaultdict
from dotenv import load_dotenv  


warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [35]:
import os, io, glob  
from collections import defaultdict  
import msoffcrypto  
import pandas as pd  
from dotenv import load_dotenv

load_dotenv()  
password = os.environ['EPIC_XL_PW']

BASE_TEMPLATE = r'L:/2026 Pilar Plant Files/AP EPIC Audit/{site}/Backup/EPIC - Received from Meaghan'  
SITES = ['LENOX', 'GVCCC', 'MEETH']


def read_protected_excel(file_path, password):  
    """Return {sheet_name: DataFrame} from a password-protected workbook."""  
    decrypted = io.BytesIO()  
    with open(file_path, 'rb') as f:  
        office_file = msoffcrypto.OfficeFile(f)  
        if office_file.is_encrypted():  
            office_file.load_key(password=password)  
            office_file.decrypt(decrypted)  
        else:  
            decrypted.write(f.read())   # handle any unprotected files  
    decrypted.seek(0)  
    return pd.read_excel(decrypted, sheet_name=None)


def load_site(site, password):  
    """Load every tab of every workbook for one site.  
    Returns (by_sheet_dict, stacked_df)."""  
    base_path = BASE_TEMPLATE.format(site=site)  
    print(f"\n=== {site} ===")

    if not os.path.isdir(base_path):  
        print(f"  !! path not found: {base_path}")  
        return {}, pd.DataFrame()

    epic_files = [  
        p for p in glob.glob(os.path.join(base_path, '*.xls*'))  
        if not os.path.basename(p).startswith('~$')  
    ]  
    print(f"  {len(epic_files)} file(s) found")

    by_sheet = defaultdict(list)  
    for file_path in epic_files:  
        filename = os.path.basename(file_path)  
        try:  
            sheets = read_protected_excel(file_path, password)  
            for sheet_name, df in sheets.items():  
                df.insert(0, 'site', site)  
                by_sheet[sheet_name].append(df)  
            print(f"  OK: {filename} ({len(sheets)} tabs)")  
        except Exception as e:  
            print(f"  ERROR in {filename}: {e}")

    by_sheet = {name: pd.concat(frames, ignore_index=True)  
                for name, frames in by_sheet.items()}

    stacked = (pd.concat(by_sheet.values(), ignore_index=True)  
               if by_sheet else pd.DataFrame())

    print(f"  Tabs found: {list(by_sheet)}")  
    for name, df in by_sheet.items():  
        print(f"    {name}: {df.shape[0]:,} rows x {df.shape[1]} cols")

    return by_sheet, stacked


# ---- run all three sites and stack them ----  
results = {site: load_site(site, password) for site in SITES}

epic_all = pd.concat(  
    [stacked for _, stacked in results.values() if not stacked.empty],  
    ignore_index=True  
)

print("\n=== SUMMARY ===")  
print(epic_all.groupby(['site']).size().to_string())  
print(f"\nCOMBINED: {epic_all.shape[0]:,} rows x {epic_all.shape[1]} cols")  


=== LENOX ===
  4 file(s) found
  OK: LHH August 2026 Implants.xlsx (1 tabs)
  OK: LHH July 2026 Implants.xlsx (1 tabs)
  OK: LHH Nov - Jun 2026 FULL.xlsx (1 tabs)
  OK: LHH September 2026 Implants.xlsx (1 tabs)
  Tabs found: ['Sheet1']
    Sheet1: 131,478 rows x 23 cols

=== GVCCC ===
  6 file(s) found
  OK: GV OR Endo and Cath Lab Implant Feb - Apr 2026.xlsx (1 tabs)
  OK: GV OR Endo and Cath Lab Implant May - June 2026.xlsx (1 tabs)
  OK: GV OR Endo and Cath Lab Implant Nov - Jan 2026.xlsx (1 tabs)
  OK: GVH Implants August 2026.xlsx (1 tabs)
  OK: GVH Jul 2026 Implants.xlsx (1 tabs)
  OK: GVH Spet 2026 Implants.xlsx (1 tabs)
  Tabs found: ['Sheet1']
    Sheet1: 19,375 rows x 23 cols

=== MEETH ===
  6 file(s) found
  OK: MEETH August Implants.xlsx (1 tabs)
  OK: MEETH Implants Mar - May.xlsx (1 tabs)
  OK: MEETH Implants Nov - Feb.xlsx (1 tabs)
  OK: MEETH July Implants.xlsx (1 tabs)
  OK: MEETH June Implants.xlsx (1 tabs)
  OK: MEETH September Implants.xlsx (1 tabs)
  Tabs found:

In [3]:
import os, glob  
import pandas as pd

AP_TEMPLATE = r'L:/2026 Pilar Plant Files/AP EPIC Audit/{site}/Backup/AP'  
SITES = ['LENOX', 'GVCCC', 'MEETH']


def load_site_ap(site, sheet_name='Voucher Drill', skiprows=3):  
    """Read the Voucher Drill tab from every AP workbook for one site."""  
    ap_path = AP_TEMPLATE.format(site=site)  
    print(f"\n=== {site} ===")

    if not os.path.isdir(ap_path):  
        print(f"  !! path not found: {ap_path}")  
        return pd.DataFrame()

    ap_files = [  
        p for p in glob.glob(os.path.join(ap_path, '*.xlsx'))  
        if not os.path.basename(p).startswith('~$')  
    ]  
    print(f"  Found {len(ap_files)} files")

    ap_data = []  
    for file in ap_files:  
        filename = os.path.basename(file)  
        try:  
            file_data = pd.read_excel(file, sheet_name=sheet_name, skiprows=skiprows)  
            file_data.insert(0, 'site', site)            # site first  
            file_data.insert(1, 'source_file', filename) # then file  
            ap_data.append(file_data)  
            print(f"  Processed: {filename} ({len(file_data):,} rows)")

        except PermissionError:  
            print(f"  SKIPPED (permission denied): {filename}")  
        except ValueError as e:  
            print(f"  SKIPPED (no '{sheet_name}' tab): {filename} -- {e}")  
        except Exception as e:  
            print(f"  ERROR in {filename}: {e}")

    if not ap_data:  
        print("  No files loaded")  
        return pd.DataFrame()

    combined = pd.concat(ap_data, ignore_index=True)  
    print(f"  Total rows: {len(combined):,} x {combined.shape[1]} cols")  
    return combined


# ---- run all three sites and stack them ----  
frames = [load_site_ap(site) for site in SITES]  
ap_all = pd.concat([df for df in frames if not df.empty], ignore_index=True)

print("\n=== SUMMARY ===")  
print(ap_all.groupby('site').size().to_string())  
print(f"\nCOMBINED: {len(ap_all):,} rows x {ap_all.shape[1]} cols")  


=== LENOX ===
  Found 3 files
  Processed: Lenox voucher detail 11 2025.xlsx (7,238 rows)
  Processed: Lenox voucher detail 12 2025.xlsx (7,932 rows)
  Processed: Lenox voucher detail YTD 9 2026.xlsx (69,934 rows)
  Total rows: 85,104 x 53 cols

=== GVCCC ===
  Found 0 files
  No files loaded

=== MEETH ===
  Found 0 files
  No files loaded

=== SUMMARY ===
site
LENOX    85104

COMBINED: 85,104 rows x 53 cols


In [4]:
acct_xwalk = pd.read_excel('L:/2026 Pilar Plant Files/Crosswalks/Manhattan Region Crosswalk.xlsx', sheet_name='all_accounts')

dept_xwalk = pd.DataFrame(  
    [  
        ('LENOX', 15652050, 'Cardiac Cath Lab',                 'Cath'),  
        ('LENOX', 15652020, 'Cardio - EPS',                     'Cath'),  
        ('GVCCC', None,     'GVH CATH & EP LABS',               'Cath'),  
        ('GVCCC', None,     'GVH Endoscopy',                    'Endo'),  
        ('GVCCC', None,     'GVH Noninvasive Cardiology',       'Cath'),  
        ('GVCCC', None,     'GVH OR',                           'OR'),  
        ('LENOX', None,     'LHH CATH & EP LABS',               'Cath'),  
        ('LENOX', None,     'LHH Endoscopy',                    'Endo'),  
        ('LENOX', None,     'LHH NONINVASIVE CARDIOLOGY',       'Cath'),  
        ('LENOX', None,     'LHH OR',                           'OR'),  
        ('LENOX', 15625000, 'Neurosurgery',                     'Cath'),  
        ('LENOX', 15611055, 'NRSG - Cardiac cath',              'Cath'),  
        ('LENOX', 15666000, 'OP Svc - Ambulatory Surgery',      'OR'),  
        ('GVCCC', 74002548, 'Op Svc Endoscopy Center - LHGV',   'Endo'),  
        ('GVCCC', 74002545, 'POS - Operating Room',             'OR'),  
        ('LENOX', 15651060, 'POS - Endoscopy',                  'Endo'),  
        ('LENOX', 15651010, 'POS - Operating Room',             'OR'),  
        ('GVCCC', 74002545, 'POS - Operating Room',             'OR'),  
        ('MEETH', 29651010, 'POS - Operating Room',             'OR'),  
        ('GVCCC', 74002562, 'Pos-Central Services',             'OR'),  
        ('MEETH', None,     'MEETH OR',                         'OR'),  
    ],  
    columns=['BU', 'Dept', 'Dept Name', 'Category']  
)

In [36]:
epic_raw = epic_all[['Location', 'Date', 'Name', 'Imp PS ID', 'Manufacturer', 'Cost', 'Number Used']]

In [23]:
ap_spec = ap_all[['Source Type', 'GL Unit', 'Year', 'Period', 'Account', 'Dept', 'Monetary Amount', 'Vendor ID', 'Vendor Name', 'Qty', 'UOM', 'Unit Price', 'Item ID', 'Description']]

depts = dept_xwalk[dept_xwalk['Dept'].notna()]
ap_dept = pd.merge(ap_spec, depts[['Dept', 'Dept Name', 'Category']], how='left', on='Dept')
ap_acct = pd.merge(ap_dept, acct_xwalk, how='left', on='Account')
ap_full = ap_acct.rename(columns={'Description_x': 'Item Desc', 'Description_y': 'Acct Desc'})
ap_cat = ap_full[ap_full['Category'].notna()]
ap_cat['Date'] = pd.to_datetime(dict(year=ap_cat['Year'], month=ap_cat['Period'], day=1))

In [66]:

accounts = (  
    ap_cat  
    .sort_values('Date')  
    .groupby(['Item ID', 'Category'], as_index=False)  
    .agg({'Account': 'last', 'Acct Desc': 'last'})  
)  

vendors = (  
    ap_cat  
    .sort_values('Date', ascending=False)  
    .groupby('Item ID', as_index=False)[['Vendor ID', 'Vendor Name']]  
    .first()  
)  

item_descs = (  
    ap_cat  
    .sort_values('Date', ascending=False)  
    .groupby('Item ID', as_index=False)[['Item Desc']]  
    .first()  
)

uoms = (  
    ap_cat  
    .sort_values('Date', ascending=False)  
    .groupby('Item ID', as_index=False)[['UOM']]  
    .first()  
)

itemids = (
    accounts
    .groupby(['Item ID', 'Category'])
    .size().reset_index(name='Count')
    .sort_values('Count', ascending=False)
    .reset_index(drop=True)
)

dupes = itemids[itemids['Count'] > 1]

if not dupes.empty:  
    print(f'{len(dupes)} Item IDs map to more than one account combination:')  
    print(dupes)
else:  
    print('All Item IDs map to exactly one account combination.')

All Item IDs map to exactly one account combination.


In [25]:
ap_new_vend = pd.merge(ap_data_cat, vendors, how='left', on='Vendor ID', suffixes=('', ' New'))

display(ap_new_vend.head())

,Source Type,GL Unit,Year,Period,Account,Dept,Monetary Amount,Vendor ID,Vendor Name,Qty,UOM,Unit Price,Item ID,Item Desc,Dept Name,Category,Acct Desc,Date,Item ID New,Vendor Name New
0,AP,LENOX,2025.0,11.0,63080.0,15651010.0,489.0,35089.0,GLOBUS MEDICAL INC,2.0,CA,244.5,244713.0,APLLICATOR SPRAY FOR HARVEST S,POS - Operating Room,OR,Supplies - Surgical,2025-11-01,154942.0,GLOBUS MEDICAL INC
1,AP,LENOX,2025.0,11.0,63080.0,15651010.0,489.0,35089.0,GLOBUS MEDICAL INC,2.0,CA,244.5,244713.0,APLLICATOR SPRAY FOR HARVEST S,POS - Operating Room,OR,Supplies - Surgical,2025-11-01,154943.0,GLOBUS MEDICAL INC
2,AP,LENOX,2025.0,11.0,63080.0,15651010.0,489.0,35089.0,GLOBUS MEDICAL INC,2.0,CA,244.5,244713.0,APLLICATOR SPRAY FOR HARVEST S,POS - Operating Room,OR,Supplies - Surgical,2025-11-01,156075.0,GLOBUS MEDICAL INC
3,AP,LENOX,2025.0,11.0,63080.0,15651010.0,489.0,35089.0,GLOBUS MEDICAL INC,2.0,CA,244.5,244713.0,APLLICATOR SPRAY FOR HARVEST S,POS - Operating Room,OR,Supplies - Surgical,2025-11-01,156378.0,GLOBUS MEDICAL INC
4,AP,LENOX,2025.0,11.0,63080.0,15651010.0,489.0,35089.0,GLOBUS MEDICAL INC,2.0,CA,244.5,244713.0,APLLICATOR SPRAY FOR HARVEST S,POS - Operating Room,OR,Supplies - Surgical,2025-11-01,161009.0,GLOBUS MEDICAL INC


In [43]:
epic_raw['Month'] = epic_raw['Date'].dt.month  
epic_raw['Year'] = epic_raw['Date'].dt.year  

epic_accts = pd.merge(epic_raw, accounts, how='left', left_on='Imp PS ID', right_on='Item ID')
epic_new_vend = pd.merge(epic_accts, vendors, how='left', on='Item ID')
epic_item_desc = pd.merge(epic_new_vend, item_descs, how='left', on='Item ID')

display(epic_item_desc.head())

,Location,Date,Name,Imp PS ID,Manufacturer,Cost,Number Used,Month,Year,Item ID,Category,Account,Acct Desc,Vendor ID,Vendor Name,Item Desc
0,LHH CATH & EP LABS,2026-08-07,Lead Solia S 53 Pm - S8002536388 - Log637425 [...,228588.0,BIOTRONIK GMBH & CO [532],340.0,1,8,2026,228588.0,Cath,63063.0,Supplies - Implants - Pacemkrs,3036.0,BIOTRONIK INC,LEAD SOLIA S 53 PM
1,LHH CATH & EP LABS,2026-08-07,Lead Solia S 53 Pm - S8002536388 - Log637425 [...,228588.0,BIOTRONIK GMBH & CO [532],340.0,1,8,2026,228588.0,OR,63063.0,Supplies - Implants - Pacemkrs,3036.0,BIOTRONIK INC,LEAD SOLIA S 53 PM
2,LHH CATH & EP LABS,2026-08-07,Envelope Tyrx Absorbable Antibacterial Med - L...,195919.0,MEDTRONIC CRM [45],995.0,1,8,2026,195919.0,Cath,63065.0,Supplies - mpl - Oth Coronary,490.0,MEDTRONIC INC,ENVELOPE TYRX ABSORBABLE ANTIB
3,LHH CATH & EP LABS,2026-08-07,Pacemaker Amvia Edge Dr-T - S1000903757 - Log6...,293234.0,BIOTRONIK GMBH & CO [532],4335.0,1,8,2026,293234.0,Cath,63063.0,Supplies - Implants - Pacemkrs,3036.0,BIOTRONIK INC,PACEMAKER AMVIA EDGE DR-T
4,LHH CATH & EP LABS,2026-08-07,Pacemaker Amvia Edge Dr-T - S1000903757 - Log6...,293234.0,BIOTRONIK GMBH & CO [532],4335.0,1,8,2026,293234.0,OR,63063.0,Supplies - Implants - Pacemkrs,3036.0,BIOTRONIK INC,PACEMAKER AMVIA EDGE DR-T


In [ ]:
ap_items_to_convert = ap_cat[ap_cat['UOM'] != 'EA']

price_counts_ap = (  
    ap_items_to_convert  
    .groupby(['Item ID', 'UOM'])['Monetary Amount']  
    .nunique()  
    .reset_index(name='n_prices')  
)

single_ap = price_counts_ap[price_counts_ap['n_prices'] == 1]  

price_counts_epic = (  
    epic_raw  
    .groupby(['Imp PS ID'])['Cost']  
    .nunique()  
    .reset_index(name='n_prices')  
)

single_epic = price_counts_epic[price_counts_epic['n_prices'] == 1]  

ap_items = single_ap['Item ID']
epic_items = single_epic['Imp PS ID']

item_ids_conversion = (  
    pd.concat([ap_items, epic_items], ignore_index=True)  
    .drop_duplicates()  
    .to_frame(name='Item ID')  
    .reset_index(drop=True)  
)  

item_ids_ap = pd.merge(item_ids_conversion, ap_cat[['Item ID', 'Monetary Amount']], how='left', on='Item ID')
item_ids_epic = pd.merge(item_ids_ap, epic_raw[['Imp PS ID', 'Cost']], how='left', left_on='Item ID', right_on='Imp PS ID')
conversion = item_ids_epic[(item_ids_epic['Imp PS ID'].notna()) & (item_ids_epic['Item ID'].notna())]
conversion['Unit'] = conversion['Monetary Amount'] / conversion['Cost']
conversion = conversion[conversion['Unit'].notna()].drop_duplicates()


,Item ID,Monetary Amount,Imp PS ID,Cost,Unit
33,1140.0,190.73,1140.0,103.19,1.848338
40,1920.0,89.00,1920.0,44.50,2.000000
58,3100.0,243.09,3100.0,81.03,3.000000
59,3100.0,243.09,3100.0,77.91,3.120139
74,3969.0,278.25,3969.0,55.65,5.000000
...,...,...,...,...,...
1634940,313501.0,5639.52,313501.0,469.96,12.000000
1634952,313525.0,2465.00,313525.0,2465.00,1.000000
1634954,313550.0,10042.20,313550.0,10042.20,1.000000
1634960,313720.0,75.00,313720.0,75.00,1.000000


In [ ]:
epic_uoms = pd.merge(epic_item_desc, uoms, how='left', on='Item ID')

epic_full = epic_uoms.copy()
epic_full['Unit Price'] = epic_full['Cost'].round(0)


display(epic_uoms)

,Location,Date,Name,Imp PS ID,Manufacturer,Cost,Number Used,Month,Year,Item ID,Category,Account,Acct Desc,Vendor ID,Vendor Name,Item Desc,UOM
0,LHH CATH & EP LABS,2026-08-07,Lead Solia S 53 Pm - S8002536388 - Log637425 [...,228588.0,BIOTRONIK GMBH & CO [532],340.00,1,8,2026,228588.0,Cath,63063.0,Supplies - Implants - Pacemkrs,3036.0,BIOTRONIK INC,LEAD SOLIA S 53 PM,EA
1,LHH CATH & EP LABS,2026-08-07,Lead Solia S 53 Pm - S8002536388 - Log637425 [...,228588.0,BIOTRONIK GMBH & CO [532],340.00,1,8,2026,228588.0,OR,63063.0,Supplies - Implants - Pacemkrs,3036.0,BIOTRONIK INC,LEAD SOLIA S 53 PM,EA
2,LHH CATH & EP LABS,2026-08-07,Envelope Tyrx Absorbable Antibacterial Med - L...,195919.0,MEDTRONIC CRM [45],995.00,1,8,2026,195919.0,Cath,63065.0,Supplies - mpl - Oth Coronary,490.0,MEDTRONIC INC,ENVELOPE TYRX ABSORBABLE ANTIB,EA
3,LHH CATH & EP LABS,2026-08-07,Pacemaker Amvia Edge Dr-T - S1000903757 - Log6...,293234.0,BIOTRONIK GMBH & CO [532],4335.00,1,8,2026,293234.0,Cath,63063.0,Supplies - Implants - Pacemkrs,3036.0,BIOTRONIK INC,PACEMAKER AMVIA EDGE DR-T,EA
4,LHH CATH & EP LABS,2026-08-07,Pacemaker Amvia Edge Dr-T - S1000903757 - Log6...,293234.0,BIOTRONIK GMBH & CO [532],4335.00,1,8,2026,293234.0,OR,63063.0,Supplies - Implants - Pacemkrs,3036.0,BIOTRONIK INC,PACEMAKER AMVIA EDGE DR-T,EA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189576,MEETH OR,2026-09-18,Arista Ent Kit (1) 2g Hemaderm And (2) Flexiti...,228130.0,BECTON DICKINSON DAVOL [5],115.00,1,9,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
189577,MEETH OR,2026-09-18,Device Vivaer Stylus 5/Bx - Sn/A - Log680060 [...,303700.0,AERIN MEDICAL INC [1770],1150.00,1,9,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
189578,MEETH OR,2026-09-18,Stent Drug Eluting Sinus Intersect Ent Propel ...,189730.0,INTERSECT ENT [895],976.50,1,9,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
189579,MEETH OR,2026-09-02,Ureteral Cath Dual Lumen 10fr 54cm - Log693360...,340.0,BOSTON SCI UROLOGY DIVISION [30],68.96,1,9,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
